# MLflow Evaluation Demo: Multi-Turn Restaurant Research Agent with Web Search

![restaurant_food_bot](./images/restaurant_food_bot.png)

This multi-turn conversational agent can be used by **Caspers Kitchens'** clients or customers, caterers, or anyone interested in researching restaurants catered by kitches such as 
**Caspers Kitchens** for the following scenarios:

* **Food allergies** — identify dishes and restaurants that accommodate specific dietary restrictions (e.g. peanut-free, gluten-free, vegan)
* **Restaurant ratings & recommendations** — discover highly rated restaurants by neighborhood, cuisine, or preference
* **Food safety inspections** — look up health inspection scores and recent violation reports for a specific restaurant
* **Menu & hours** — find current operating hours, menus, and vegetarian or allergen-friendly options
* **Personalized recommendations** — get synthesized advice across multiple turns, with the agent remembering your preferences throughout the conversation

This notebook demonstrates MLflow 3.10 evaluation capabilities using a web-search-augmented multi-turn agent:

1. Set up session-level judges for multi-turn conversation evaluation
2. Use `mlflow.genai.evaluate()` to evaluate full conversations
3. Evaluate **three dimensions**: coherence, context retention, and search quality
4. Inspect traces and results in the MLflow UI

## What You'll Learn

- How to create session-level judges with the `{{ conversation }}` template
- How a tool-use loop (web search via Tavily) is traced automatically
- How to evaluate a stateless-search agent for context carryover

---

In [ ]:
import mlflow
print(mlflow.__version__)

## Setup: Import Dependencies

In [ ]:
import os
import mlflow
import pandas as pd
from pathlib import Path


from devconnect.config import AgentConfig
from devconnect.restaurant_research_bot.restaurant_research_agent_cls import RestaurantResearchAgent
from devconnect.restaurant_research_bot.scenarios import get_scenario_food_safety
from devconnect.restaurant_research_bot.prompts import (
    SYSTEM_PROMPT_NAME,
    COHERENCE_JUDGE_NAME,
    CONTEXT_RETENTION_JUDGE_NAME,
    SEARCH_QUALITY_JUDGE_NAME,
)

# Load environment variables from .env if present
try:
    from dotenv import load_dotenv
    env_file = Path(".env")
    if env_file.exists():
        load_dotenv(env_file)
        print(f"Loaded environment variables from {env_file.absolute()}")
    else:
        print("No .env file found. Set OPENAI_API_KEY and TAVILY_API_KEY manually.")
except ImportError:
    print("python-dotenv not installed. Set environment variables manually.")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

## Configuration

In [ ]:
# Provider and model configuration
PROVIDER = "databricks"            # or "databricks"
AGENT_MODEL = "databricks-gpt-5-mini"
JUDGE_MODEL = "databricks-5-mini"
TEMPERATURE = 0.2
EXPERIMENT_NAME = "restaurant-research-bot"

print("Configuration:")
print(f"  Provider:    {PROVIDER}")
print(f"  Agent Model: {AGENT_MODEL}")
print(f"  Judge Model: {JUDGE_MODEL}")
print(f"  Experiment:  {EXPERIMENT_NAME}")

In [ ]:
# Verify required credentials are set
tavily_key = os.environ.get("TAVILY_API_KEY", "")

print(f"TAVILY_API_KEY  set: {'yes' if tavily_key else 'NO - required for web search'}")

## Step 1: Setup MLflow Tracking

In [ ]:
mlflow.openai.autolog()

using_databricks_mlflow = False  # Set to True if you're using Databricks MLflow on Databricks Workspace
mlflow.set_tracking_uri("databricks")
EXPERIMENT_NAME_FULL = f"/Users/jules@databricks.com/{EXPERIMENT_NAME}"
mlflow.set_experiment(EXPERIMENT_NAME_FULL)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

## Step 2: Initialize the Restaurant Research Agent

The agent uses a tool-use loop: when it needs current data (hours, ratings, menus),
it calls `web_search()` via Tavily, feeds the results back, and continues.

Each `handle_message()` call produces a `CHAT_MODEL` trace with `TOOL` child spans
for every web search performed.

In [ ]:
# Build judge model URI — MLflow make_judge() requires "<provider>:/<model>" format

judge_model_uri = f"databricks:/{JUDGE_MODEL}"

config = AgentConfig(
    model=AGENT_MODEL,
    provider=PROVIDER,
    temperature=TEMPERATURE,
    mlflow_experiment=EXPERIMENT_NAME,
)

agent = RestaurantResearchAgent(config=config, judge_model=judge_model_uri, debug=True)

print("\nAgent initialised")
print(f"  Provider: {config.provider}")
print(f"  Model:    {config.model}")
print(f"  Judge:    {judge_model_uri}")

## Step 3: Run a Conversation Scenario

The **Food Safety Research** scenario tests whether the agent:
- Searches appropriately for current inspection data
- Resolves implicit references ("that restaurant" → Nopa) in search queries
- Synthesises findings at the end without re-searching

In [ ]:
scenario = get_scenario_food_safety()

print(f"Scenario:   {scenario['name']}")
print(f"Session ID: {scenario['session_id']}")
print(f"Turns:      {len(scenario['messages'])}")
print("\nMessages:")
for i, msg in enumerate(scenario['messages'], 1):
    print(f"  Turn {i}: {msg}")

In [ ]:
# Run the conversation inside an MLflow run.
# All handle_message() traces produced here share the run_id,
# which evaluate_session() uses to find them.
with mlflow.start_run(run_name=scenario["name"]) as run:
    agent.run_conversation(scenario["messages"], scenario["session_id"])
    run_id = run.info.run_id

print(f"\nConversation complete. Run ID: {run_id}")

---

# Evaluation Showcase: MLflow Session-Level Judges

We evaluate the conversation along **three dimensions**:

| Judge | Type | What it measures |
|---|---|---|
| `conversation_coherence` | `bool` | Does the conversation flow logically? |
| `context_retention` | `excellent/good/fair/poor` | Does the agent remember prior constraints? |
| `search_quality` | `necessary/unnecessary/skipped` | Did the agent search at the right times? |

## Step 4: View Judge Instructions

All three judges use the `{{ conversation }}` template — this is what makes them
**session-level**: MLflow aggregates all turns and passes the full conversation to each judge.

In [ ]:
for name in [
    SYSTEM_PROMPT_NAME,
    COHERENCE_JUDGE_NAME,
    CONTEXT_RETENTION_JUDGE_NAME,
    SEARCH_QUALITY_JUDGE_NAME,
]:
    prompt = mlflow.genai.load_prompt(name)
    print("=" * 70)
    print(f"{name}  (v{prompt.version})")
    print("=" * 70)
    print(prompt.template)
    print()

print("All three judge prompts contain {{ conversation }} -- session-level!")

# Use the prompts in the MLflow UI:
print("View the prompts in the Databricks UI")

## Step 5: Search for Traces with `mlflow.search_traces()`

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

session_traces = mlflow.search_traces(
    locations=[experiment.experiment_id],
    filter_string=f"metadata.`mlflow.trace.session` = '{scenario['session_id']}'"
)

print(f"Found {len(session_traces)} traces for session '{scenario['session_id']}'")
print("Each trace = one conversation turn (with tool spans for web searches)")

display(session_traces[["request_time", "request", "response"]].sort_values(
    by="request_time", ascending=True
))

## Step 6: Evaluate with `mlflow.genai.evaluate()`

`evaluate_session()` internally calls `mlflow.genai.evaluate()` with all three judges.
Each session-level judge receives the **full conversation** via `{{ conversation }}`
rather than evaluating individual turns.

In [ ]:
results = agent.evaluate_session(scenario["session_id"], run_id)

print("\nEvaluation complete.")

## Step 7: Inspect Results

In [ ]:
coh  = results["coherence"]
ctx  = results["context_retention"]
srch = results["search_quality"]

print("=" * 50)
print(f"Session:  {results['session_id']}")
print(f"Traces:   {results['num_traces']}")
print("=" * 50)
print(f"\nCoherence:         {'PASS' if coh['passed'] else 'FAIL'}  ({coh['feedback_value']})")
if coh["rationale"]:
    print(f"  {coh['rationale']}")

print(f"\nContext Retention: {str(ctx['feedback_value']).upper()}")
if ctx["rationale"]:
    print(f"  {ctx['rationale']}")

print(f"\nSearch Quality:    {str(srch['feedback_value']).upper()}")
if srch["rationale"]:
    print(f"  {srch['rationale']}")

In [ ]:
# View in MLflow UI
print("View results in MLflow UI:")
print("  mlflow ui")
print(f"  http://localhost:5000  ->  '{EXPERIMENT_NAME}' experiment")
print(f"  Chat Sessions tab -> {scenario['session_id']}")

---

## What Just Happened?

1. **`RestaurantResearchAgent`** ran a 4-turn conversation with a tool-use loop — each
   web search appears as a child `TOOL` span inside the `CHAT_MODEL` trace.

2. **`mlflow.update_current_trace()`** tagged every trace with the session ID, allowing
   MLflow to group all turns as one session.

3. **`make_judge()`** created three session-level judges. The `{{ conversation }}` template
   variable tells MLflow to aggregate all turns before scoring.

4. **`mlflow.genai.evaluate()`** fanned out all traces to the three judges simultaneously.

5. **`search_quality`** is unique to this agent: it verifies the agent searched when it
   needed to (`necessary`), didn't over-search (`unnecessary`), and didn't silently skip
   searches it should have made (`skipped`).